In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
webArtists         = MusicDBData(path=permDir, fname="{0}SearchedForWebArtists".format(db.lower()))
webArtistsData     = MusicDBData(path=permDir, fname="{0}SearchedForWebArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistsData   = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsData".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
searchWebArtists   = mio.data.getSearchWebArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Web Artists Search:        {0}".format(len(webArtists.get())))
print("   Web Artists Search Data:   {0}".format(len(webArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist Search Data:  {0}".format(len(localArtistsData.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Search Web Artists:        {0}".format(len(searchWebArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Web Artists Search:        4522
   Web Artists Search Data:   269714
   Local Artist Search:       164237
   Local Artist Search Data:  0
   Global Master Search:      34949
   Global Master Search Data: 0
   Search Artists:            34949
   Search Web Artists:        135378
   Errors:                    263
   Known Summary IDs:         0


# Search For New Artists

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
from musicdb import PanDBIO
from gate import IOStore

pdbio = PanDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

ios     = IOStore()
mdbio   = ios.get(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

del pdbio
del mmeDF
del refData
del mbIDData

#   Artist Names To Get:          793373

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

## Run @ 3-4 PM every day

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(20)
    n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

In [ ]:

del searchedForErrors['124024766635548005270309261423453479764']
del searchedForErrors['237748617780344156692083260097110805341']
del searchedForErrors['231184520235867051473735912378686360891']
del searchedForErrors['2875464283021047873405831377507163865']
del searchedForErrors['219261416572199174528917937125268559566']
del searchedForErrors['191092118056524447748174293367409296013']
errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

# Website Download

In [ ]:
from apiutils import WebIO
from ioutils import HTMLIO, FileIO
from bs4 import element
wio = WebIO()
hio = HTMLIO()
io  = FileIO()

## Starter

In [ ]:
modValRefs  = {}
modValLinks = []
for modVal in modVals:
    url = "https://www.setlist.fm/artist/browse/{0}/1.html".format(modVal)
    savename = "{0}.p".format(modVal)
    retval = wio.get(url)
    print(url)
    io.save(idata=retval.getData(), ifile=savename)
    wio.sleep(5)

In [ ]:
modValListRefs = {}
for modVal in modVals:
    savename = "{0}.p".format(modVal)
    bsdata = hio.get(io.get(savename))
    
    submodListDiv  = bsdata.find("div", {"id": "ide"})
    submodListRefs = [li.find('a') for li in submodListDiv.findAll("li")] if isinstance(submodListDiv, element.Tag) else []
    submodListRefs = {ref.get('href'): ref.text for ref in submodListRefs if (isinstance(ref, element.Tag) and "/setlists/" in ref.get('href'))}
    modValListRefs.update(submodListRefs)

## Download ModVal Refs

In [ ]:
from string import ascii_lowercase
modVals = [ch for ch in ascii_lowercase] + ['0-9']

modValRefs = {}
for modVal in modVals:
    savename = "{0}.p".format(modVal)
    bsdata = hio.get(io.get(savename))
    modListUL   = bsdata.find("ul", {"class": "row"})
    modListRefs = [li.find('a') for li in modListUL.findAll("li")] if isinstance(modListUL, element.Tag) else []
    modValRefs.update({ref.get('href'): ref.text for ref in modListRefs if isinstance(ref, element.Tag)})
modValRefs = Series(modValRefs)

searchedForWebArtists = webArtists.get()
modValRefsToGet = modValRefs[~modValRefs.index.isin(searchedForWebArtists.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   All ModVal Refs:    {0}".format(modValRefs.shape[0]))
print("   Known ModVal Refs:  {0}".format(len(searchedForWebArtists)))
print("   ModVal Refs To Get: {0}".format(len(modValRefsToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localWebArtistsDict  = webArtists.get()
modValListRefs = {}

for i,(ref,name) in enumerate(modValRefsToGet.iteritems()):
    if localWebArtistsDict.get(ref) is not None:
        continue
        
    url = ref.replace("../../../", "https://www.setlist.fm/")
    try:
        print("Getting {0: <50}".format(url), end="\t")
        response = wio.get(url)
    except:
        print("ERROR Downloading! Stopping.")
        break
        

    try:
        bsdata = hio.get(response.getData())
    except:
        print("ERROR Creating BeautifulSoup! Stopping.")
        break
        
    try:
        submodListDiv  = bsdata.find("div", {"id": "ide"})
        submodListRefs = [li.find('a') for li in submodListDiv.findAll("li")] if isinstance(submodListDiv, element.Tag) else []
        submodListRefs = {ref.get('href'): ref.text for ref in submodListRefs if (isinstance(ref, element.Tag) and "/setlists/" in ref.get('href'))}
    except:
        print("ERROR Getting Refs From BeautifulSoup! Stopping.")
        break
        
    print(len(submodListRefs))
    modValListRefs.update(submodListRefs)    
    localWebArtistsDict[ref] = True
    wio.sleep(5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
        webArtists.save(data=localWebArtistsDict)
        
        previousData = webArtistsData.get()
        print("Found {0} Previous Web Links".format(len(previousData)))
        print("Found {0} New Web Links".format(len(modValListRefs)))
        newData = {**previousData, **modValListRefs}
        print("Found {0} Total Web Links".format(len(newData)))
        webArtistsData.save(data=newData)
        modValListRefs = {}
        
        print("="*150)
        wio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
webArtists.save(data=localWebArtistsDict)

previousData = webArtistsData.get()
print("Found {0} Previous Web Links".format(len(previousData)))
print("Found {0} New Web Links".format(len(modValListRefs)))
newData = {**previousData, **modValListRefs}
print("Found {0} Total Web Links".format(len(newData)))
webArtistsData.save(data=newData)


In [ ]:
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
webArtists.save(data=localWebArtistsDict)

previousData = webArtistsData.get()
print("Found {0} Previous Web Links".format(len(previousData)))
print("Found {0} New Web Links".format(len(modValListRefs)))
newData = {**previousData, **modValListRefs}
print("Found {0} Total Web Links".format(len(newData)))
webArtistsData.save(data=newData)


# Download Artist Refs

In [12]:
from pandas import merge
from apiutils import WebIO
from ioutils import HTMLIO, FileIO
from bs4 import element
wio = WebIO()
hio = HTMLIO()
io  = FileIO()

In [6]:
webArtistsDataDict = webArtistsData.get()
setlistWebRefs = DataFrame(Series(webArtistsDataDict, name="ArtistName"))
setlistWebRefs.index.name = "SetListFM"
setlistWebRefs = setlistWebRefs.reset_index()
setlistWebRefs["ID"] = setlistWebRefs["SetListFM"].apply(lambda x: x.split("-")[-1][:-5])
def fixName(x):
    vals = x.split(", ")
    if len(vals) == 2:
        return " ".join([vals[1], vals[0]])
    return x
setlistWebRefs["ArtistName"] = setlistWebRefs["ArtistName"].apply(fixName)

In [10]:
useWebArtists   = True
useKnownArtists = False

if useKnownArtists is True:
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    panData = mmeDF["ArtistName"].reset_index().copy(deep=True)
    tmp   = mmeDF[mmeDF["SetListFM"].notna()][["ArtistName", "SetListFM"]]
    known = Series(tmp["ArtistName"].values, index=tmp["SetListFM"].values)
    knownSetListFMArtists = merge(panData, setlistWebRefs, on="ArtistName", how='left')    
    allSetListFM     = knownSetListFMArtists[knownSetListFMArtists["SetListFM"].notna()]
    localArtistsDict = localArtists.get()
    artistNamesToGet = allSetListFM[~allSetListFM["ID"].isin(localArtistsDict.keys())]
elif useWebArtists is True:
    allSetListFM     = setlistWebRefs
    localArtistsDict = localArtists.get()
    artistNamesToGet = allSetListFM[~allSetListFM["ID"].isin(localArtistsDict.keys())]

print("# {0} Search Results".format(db))
print("#   All Artist Refs:    {0}".format(allSetListFM.shape[0]))
print("#   Known Artist Refs:  {0}".format(len(localArtistsDict)))
print("#   Artist Refs To Get: {0}".format(len(artistNamesToGet)))

# SetListFM Search Results
#   All Artist Refs:    269714
#   Known Artist Refs:  164237
#   Artist Refs To Get: 108379


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "10:50am")
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
localArtistsDataDict = localArtistsData.get()
searchedForErrors    = errors.get()
nErr = []

for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):        
    artistID   = row["ID"]
    artistName = row["ArtistName"]
    artistURL  = row["SetListFM"].replace("../../../", "https://www.setlist.fm/")
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    print("Getting {0: <50}".format("{0} ({1})".format(artistName,artistID)), end="\t")
    
    try:
        response = wio.get(artistURL)
    except:
        print("ERROR Downloading! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue

    try:
        bsdata = hio.get(response.getData())
    except:
        print("ERROR Creating BeautifulSoup! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue
        
    try:
        artistLinks  = bsdata.find("div", {"class": "artistLinks"})
        externalRefs = {ref.get('href'): ref.text.strip() for ref in artistLinks.findAll("a")} if isinstance(artistLinks, element.Tag) else {}
        mbDiv = bsdata.find("div", {"class": "info"})
        mbid  = mbDiv.find('span').text if isinstance(mbDiv, element.Tag) else None
        artistData = {"MBID": mbid, "Refs": externalRefs}
    except:
        print("ERROR Getting Refs From BeautifulSoup! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue
        
    
    print("{0}  ({1})".format(artistData["MBID"], len(artistData["Refs"])))
    localArtistsDict[artistID] = artistName
    localArtistsDataDict[artistID] = artistData
    nErr = []
    wio.sleep(6)
    n += 1
    
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        localArtistsData.save(data=localArtistsDataDict)
        errors.save(data=searchedForErrors)
        print("="*150)
        wio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
    localArtists.save(data=localArtistsDict)
    localArtistsData.save(data=localArtistsDataDict)
    errors.save(data=searchedForErrors)
    if len(nErr) > 0:
        for artistID in nErr:
            print("del searchedForErrors['{0}']".format(artistID))
        print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM Artist Data] Start    ==> Time Is 2022-05-24 12:34:45
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-05-24 21:00:00 <====
   ====> Will Terminate Process 8 Hours and 25 Minutes From Now
Getting A 5 Pasos (7bd9fa80)                              	359db908-7960-4f5f-a5e2-47e2830a4c4d  (1)
Getting A B & The Sea (23df7857)                          	c82215e2-56fd-45a6-8d39-4bb3679bc6a4  (1)
Getting A Bad Girl's Dream (33f48c9d)                     	983bee53-1f29-4a82-89e1-ce321783f8ed  (1)
Getting A Banca (23dddc9f)                                	e6565d98-67aa-4e47-a55b-41f8a3632623  (1)
Getting A Band In Ship (53c9c759)                         	f0f64cd5-f333-47c3-a1e5-81a342f9bcda  (2)
Getting A Band of Bitches (63dac673)                      	377c229e-a9ff-4d57-8e8b-668d4d27ddbc  (2)
Getting A Banda da Loba (bf4dd12)                         	0efbd1fd-8d07-4695-bbda-0801a51228db  (2)
Getting

In [98]:
##### from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df



lad = localArtistsData.get()
df  = getResponseDataFrame(lad)
nameData = Series(localArtists.get())
nameData.name = "Name"
df = df.join(nameData)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchWebArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]    
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchWebArtistData(data=searchDF)
    print("Done")

Found 83 New Artists
Found 135295 Previous Artists
Found 135378 Total Results
Found 135378 Unique Results
Saving Data
Done


In [99]:
localArtistsData.save(data={})

In [94]:
errs = {k: v for k,v in searchedForErrors.items() if len(k) > 10}

In [95]:
errors.save(data=errs)